# 🪨 Rosetta — gerar SQL Databricks de uma CDS view

Preencha o widget **`ddlname`** (ex.: `BSAS_DDL`, `I_ADDRESS`, `/AIF/C_INTERFACESTATISTICS`)
e rode as células em ordem. O notebook mostra a árvore de dependências, as tabelas físicas
que compõem a view, gera o SQL Databricks e salva em `ddl/<DDLNAME>/`.

Toda a lógica está no pacote `rosetta` (pasta `src/`) — este notebook é só a interface.
Para mexer em parser, tradução ou geração, edite os módulos, não o notebook.

---
### 🔒 Somente leitura
O SQL gerado **nunca é executado**: sai com o `CREATE OR REPLACE VIEW` comentado e é
apenas gravado como arquivo de texto. Toda consulta ao Spark passa por
`rosetta.seguranca.sql_leitura`, que recusa qualquer statement que não comece com
`SELECT`/`WITH`/`DESCRIBE`/`SHOW`/`EXPLAIN`. Nada é criado nem alterado no catálogo.

## 1. 🎛️ Widgets

In [0]:
dbutils.widgets.removeAll()

dbutils.widgets.text("ddlname", "I_ADDRESS", "01 | DDL Name da CDS View")
dbutils.widgets.dropdown("seguir_associations", "Nao", ["Sim", "Nao"],
                         "02 | Seguir associations na árvore")
dbutils.widgets.text("max_profundidade", "15", "03 | Profundidade máxima da árvore")
dbutils.widgets.dropdown("gravar_arquivos", "Sim", ["Sim", "Nao"],
                         "04 | Gravar em ddl/<DDLNAME>/")

dbutils.widgets.text("catalog_raw",    "platform_dev",       "10 | Catalog origem")
dbutils.widgets.text("schema_raw",     "sap_s4_nc2_raw",     "11 | Schema origem (raw)")
dbutils.widgets.text("tabela_ddl",     "tab_ddddlsrc",       "12 | Tabela DDDDLSRC")
dbutils.widgets.text("tabela_dep",     "ddldependency",      "13 | Tabela DDLDEPENDENCY")
dbutils.widgets.text("catalog_target", "platform_dev",       "14 | Catalog destino")
dbutils.widgets.text("schema_target",  "sap_s4_nc2_replica", "15 | Schema destino")

print("🎛️  Widgets criados.")

## 2. 📦 Carregar o pacote `rosetta`

`autoreload` faz o notebook reler os módulos a cada célula — você edita um arquivo em
`src/rosetta/` e a mudança vale na próxima execução, sem reiniciar o cluster.

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import sys
from pathlib import Path

# Sobe a árvore de diretórios até achar src/rosetta/ e coloca src/ no sys.path.
_aqui = Path.cwd()
for _cand in [_aqui, *_aqui.parents]:
    if (_cand / "src" / "rosetta" / "__init__.py").exists():
        RAIZ = _cand
        break
else:
    raise FileNotFoundError(f"Raiz do repositório não encontrada a partir de {_aqui}")

if str(RAIZ / "src") not in sys.path:
    sys.path.insert(0, str(RAIZ / "src"))

import rosetta
from rosetta import Config, Contexto

PASTA_DDL = RAIZ / "ddl"

print(f"📦 rosetta {rosetta.__version__}")
print(f"📁 raiz do repo : {RAIZ}")
print(f"📁 saída do SQL : {PASTA_DDL}")

## 3. 📖 Configuração e índice de fontes

In [0]:
def _w(nome, padrao=""):
    try:
        return (dbutils.widgets.get(nome) or "").strip() or padrao
    except Exception:
        return padrao

DDLNAME          = _w("ddlname").upper()
SEGUIR_ASSOC     = _w("seguir_associations", "Nao") == "Sim"
MAX_PROFUNDIDADE = int(_w("max_profundidade", "15"))
GRAVAR           = _w("gravar_arquivos", "Sim") == "Sim"

CFG = Config.de_widgets(dbutils)
print(CFG.resumo())
print(f"\n🎯 DDL Name: '{DDLNAME}'")
print(f"🔗 Seguir associations: {SEGUIR_ASSOC}  |  📏 Profundidade: {MAX_PROFUNDIDADE}")
print(f"💾 Gravar arquivos: {GRAVAR}")

O índice `ddlname → source` é carregado **uma vez por sessão** (consulta única, ~114 mil
linhas). Depois disso a árvore de dependências roda inteiramente em memória — sem disparar
um job Spark por nó, que era o que travava em views largas.

In [0]:
# Reaproveita o índice se já estiver carregado nesta sessão (trocar o ddlname
# no widget e rodar de novo não deve recarregar 114 mil fontes).
if "CTX" not in globals() or CTX.cfg != CFG:
    CTX = Contexto(spark, CFG, raiz_ddl=PASTA_DDL)
else:
    print(f"📇 Índice já em memória: {len(CTX.indice):,} entradas")

## 4. 🌲 Árvore de dependências e tabelas físicas

In [0]:
if not CTX.existe(DDLNAME):
    print(f"❓ '{DDLNAME}' não existe em {CFG.fqn_ddl}.")
    print("   Confira o widget 'ddlname' — o nome precisa ser o DDL name, não o nome da entidade.")
    RES = None
else:
    RES = CTX.traduzir(
        DDLNAME,
        seguir_assoc=SEGUIR_ASSOC,
        max_profundidade=MAX_PROFUNDIDADE,
        gravar=False,          # a gravação acontece na seção 6, depois de você ver o SQL
    )

    print(f"🌲 Árvore de {DDLNAME} (seguir associations: {SEGUIR_ASSOC})")
    print("-" * 78)
    print(RES.arvore.texto())
    print("-" * 78)
    print(f"🗄️  Tabelas físicas na base: {len(RES.arvore.tabelas_fisicas)}")
    for t in sorted(RES.arvore.tabelas_fisicas):
        print(f"   • {CFG.fqn_raw}.{t.lower()}")
    if RES.arvore.truncadas:
        print(f"\n❌ Dependências truncadas ({len(RES.arvore.truncadas)}): "
              + ", ".join(sorted(RES.arvore.truncadas)))
    if RES.arvore.nao_encontradas:
        print(f"❓ Não encontradas: " + ", ".join(sorted(RES.arvore.nao_encontradas)))

## 5. 📝 SQL Databricks gerado

In [0]:
if RES is not None:
    print("=" * 78)
    print(f"  {DDLNAME}  →  SQL Databricks")
    print("=" * 78)
    print()
    print(RES.sql)
    print()
    print("=" * 78)
    if RES.avisos:
        print(f"  ⚠️  AVISOS ({len(RES.avisos)}) — revisar antes de usar")
        print("=" * 78)
        for i, a in enumerate(RES.avisos, 1):
            print(f"  {i:2}. {a}")
    else:
        print("  ✅ Nenhum aviso — tradução direta.")
    print("=" * 78)

## 6. 💾 Salvar em `ddl/<DDLNAME>/`

Grava quatro arquivos: o `.sql`, a árvore, os avisos (quando houver) e um `metadata.json`
com entidade, tipo, tabelas físicas e contagens. Isso é escrita de **arquivo no repositório** —
o catálogo do Databricks continua intocado.

In [0]:
from rosetta import salvar_artefatos

if RES is not None:
    # Reusa o resultado da seção 4 — não reprocessa a árvore só para gravar.
    ART = salvar_artefatos(
        raiz_ddl=PASTA_DDL,
        ddlname=DDLNAME,
        sql=RES.sql,
        avisos=RES.avisos,
        view=RES.view,
        arvore=RES.arvore,
        gravar=GRAVAR,
    )
    RES.artefatos = ART

    print(ART.resumo())
    if not GRAVAR:
        print("\n   (widget 'gravar_arquivos' = Nao — nada foi escrito em disco)")

## 7. 📋 Resumo

In [0]:
if RES is None:
    print("⏹️  Nada processado — ajuste o widget 'ddlname'.")
else:
    print("=" * 78)
    print(f"  📋 {RES.selo()}")
    print("=" * 78)
    print(f"  entidade CDS : {RES.view.nome_entidade}")
    print(f"  tipo         : {RES.view.tipo}  |  estilo: {RES.view.estilo}")
    print(f"  campos       : {len(RES.view.campos)}")
    print(f"  associations : {len(RES.view.associacoes)}  |  joins: {len(RES.view.joins)}")
    if RES.view.blocos_uniao:
        print(f"  branches UNION: {len(RES.view.blocos_uniao)}")
    print(f"  tabelas base : {len(RES.arvore.tabelas_fisicas)}")
    print("=" * 78)
    print("\n🔒 Nada foi criado no catálogo. O SQL salvo tem o CREATE comentado.")
    print("👉 Para outra view: mude o widget 'ddlname' e rode a partir da seção 3.")